In [416]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import signal
from scipy.io import loadmat
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from scipy.signal import medfilt
from torch.utils.data import WeightedRandomSampler
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader

In [417]:
def normalise(data, train_reps):
    """
    Normalise function rewritten to support different numbers of channels
    """
    x = [np.where(data.values[:, data.shape[1] - 2] == rep) for rep in train_reps]
    indices = np.squeeze(np.concatenate(x, axis=-1))
    train_data = data.iloc[indices, :]
    train_data = train_data.reset_index(drop=True)

    scaler = StandardScaler(with_mean=True,
                            with_std=True,
                            copy=False).fit(train_data.iloc[:, :data.shape[1] - 2])

    scaled = scaler.transform(data.iloc[:, :data.shape[1] - 2])
    normalised = pd.DataFrame(scaled)
    normalised['stimulus'] = data['stimulus']
    normalised['repetition'] = data['repetition']
    return normalised


def filter_data(data, f, butterworth_order=4, btype='lowpass'):
    """
    Filter Data function rewritten to support different numbers of channels
    """
    emg_data = data.values[:, :data.shape[1] - 2]

    f_sampling = 2000
    nyquist = f_sampling / 2
    if isinstance(f, int):
        fc = f / nyquist
    else:
        fc = np.array(f, dtype=float) / nyquist
    b, a = signal.butter(butterworth_order, fc, btype=btype)
    transpose = emg_data.T.copy()
    for i in range(len(transpose)):
        transpose[i] = (signal.lfilter(b, a, transpose[i]))
    filtered = pd.DataFrame(transpose.T)
    filtered['stimulus'] = data['stimulus']
    filtered['repetition'] = data['repetition']
    return filtered

def windowing(data, reps, gestures, win_len, win_stride):
    """
    Windowing function rewritten to support different numbers of channels
    """
    if reps:
        x = [np.where(data.values[:, data.shape[1] - 1] == rep) for rep in reps]
        indices = np.squeeze(np.concatenate(x, axis=-1))
        data = data.iloc[indices, :]
        data = data.reset_index(drop=True)
    if gestures:
        x = [np.where(data.values[:, data.shape[1] - 2] == move) for move in gestures]
        indices = np.squeeze(np.concatenate(x, axis=-1))
        data = data.iloc[indices, :]
        data = data.reset_index(drop=True)
    
    idx = [i for i in range(win_len, len(data), win_stride)]
    X = np.zeros([len(idx), win_len, len(data.columns) - 2])
    y = np.zeros([len(idx), ])
    reps = np.zeros([len(idx), ])

    for i, end in enumerate(idx):
        start = end - win_len
        X[i] = data.iloc[start:end, 0:data.shape[1] - 2].values
        mid = start + win_len // 2
        y[i] = data.iloc[mid, data.shape[1] - 2]

        reps[i] = data.iloc[end, data.shape[1] - 1]

    return X, y, reps


def notch_filter(data, f0, Q, fs=2000):
    """
    Notch Filter function rewritten to support different numbers of channels
    """
    emg_data = data.values[:, :data.shape[1] - 2]

    b, a = signal.iirnotch(f0, Q, fs)
    transpose = emg_data.T.copy()

    for i in range(len(transpose)):
        transpose[i] = (signal.lfilter(b, a, transpose[i]))

    filtered = pd.DataFrame(transpose.T)
    filtered['stimulus'] = data['stimulus'].values
    filtered['repetition'] = data['repetition'].values

    return filtered

def load_data(path):
    mat = loadmat(path) 
    emg = mat['emg']

    print(f"subject #{mat['subject']}") # subject number
    print(f"exercise #{mat['exercise']}") # exercise number

    data = pd.DataFrame(mat['emg'])
    data['stimulus'] = mat['restimulus']
    data['repetition'] = mat['repetition']
    return data

def preprocessing(path):
    data = load_data(path)
    return preprocessing_internals(data)

def preprocessing_internals(data):
    emg_low = filter_data(data=data, f=450, butterworth_order=4, btype='lowpass')

    emg_notch = notch_filter(data=emg_low,f0=60,Q=30,fs=2000)

    gestures = data['stimulus'].unique().tolist() 
    train_reps, test_reps = train_test_split(
        data['repetition'].unique().tolist(),
        test_size=0.3,
        random_state=42,
        shuffle=True
    )

    emg_norm = normalise(data = emg_notch, train_reps=train_reps)

    win_len = 200    
    win_stride = 50

    X_train, y_train, r_train = windowing(emg_norm, train_reps, gestures, win_len, win_stride)
    X_test, y_test, r_test = windowing(emg_norm, test_reps, gestures, win_len, win_stride)
    overlap = np.intersect1d(r_train, r_test)
    print(pd.Series(y_train).value_counts())
    print(pd.Series(y_test).value_counts())

    if len(overlap) != 0:
        raise Exception(f'repetitions leaked between train and test for repetition {overlap}')
    
    class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weights_dict = dict(zip(np.unique(y_train), class_weights))

    return X_train, y_train, X_test, y_test, class_weights_dict

# path = "../NinaProData"

def multi_preprocess(exercise_number, path):
    num_subjects = 27
    x_train_list, y_train_list, x_test_list, y_test_list = [], [], [],[]
    for i in range(num_subjects):
        base_path = f"{path}/s{i + 1}/S{i + 1}_A1_E{exercise_number}.mat"
        data = load_data(base_path)
        x_train, y_train, x_test, y_test, _cw = preprocessing_internals(data)
        x_train_list.append(x_train)
        y_train_list.append(y_train)
        x_test_list.append(x_test)
        y_test_list.append(y_test)

    x_train = np.concatenate(x_train_list)
    y_train = np.concatenate(y_train_list)
    x_test = np.concatenate(x_test_list)
    y_test = np.concatenate(y_test_list)

    class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weights_dict = dict(zip(np.unique(y_train), class_weights))
    x_test = np.transpose(x_test, (0, 2, 1))
    x_train = np.transpose(x_train, (0, 2, 1))
    
    return x_train, y_train, x_test, y_test, class_weights_dict
class NinaProDataset(torch.utils.data.Dataset):
  def __init__(self, x, y):
    self.X = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.long)

  def __len__(self):
    return len(self.X)
  
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [418]:
class GLU(nn.Module):
    def __init__(self, in_size):
        super().__init__()
        self.linear1 = nn.Linear(in_size, in_size)
        self.linear2 = nn.Linear(in_size, in_size)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # x: (batch, time, channels) or (batch, seq_len, features)
        a = F.relu(self.linear1(x))
        b = torch.sigmoid(self.linear2(x))
        x = a * b
        return self.dropout(x)

In [419]:
class ConvGLUBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=5, padding=2):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=padding)
        self.bn = nn.BatchNorm1d(out_ch)
        # gating branch uses 1x1 conv on input
        self.gate = nn.Conv1d(in_ch, out_ch, kernel_size=1)
        self.res_proj = None
        if in_ch != out_ch:
            self.res_proj = nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x):
        # x: (batch, channels, time)
        y = self.conv(x)
        y = self.bn(y)
        g = torch.sigmoid(self.gate(x))
        out = F.relu(y * g)
        res = x if self.res_proj is None else self.res_proj(x)
        return F.relu(out + res)

In [420]:
X_train, y_train, X_test, y_test, class_weights = preprocessing('s1/S1_A1_E2.mat')
glu = GLU(10)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
output = glu(X_train_tensor)
print("GLU output shape:", output.shape)

subject #[[1]]
exercise #[[2]]
0.0     490
17.0     57
9.0      50
12.0     50
1.0      49
6.0      47
16.0     47
13.0     46
2.0      45
4.0      45
7.0      43
3.0      42
10.0     41
8.0      39
5.0      36
11.0     36
15.0     34
14.0     30
Name: count, dtype: int64
0.0     1302
6.0       26
17.0      24
3.0       22
12.0      22
16.0      21
10.0      20
2.0       19
7.0       19
8.0       19
9.0       19
11.0      19
15.0      19
1.0       18
13.0      17
5.0       15
4.0       14
14.0      11
Name: count, dtype: int64
GLU output shape: torch.Size([1227, 200, 10])


In [421]:
class ResGLUClassifier(nn.Module):
    def __init__(self, in_channels, num_classes, base_channels=32):
        super().__init__()
        self.stem = nn.Conv1d(in_channels, base_channels, kernel_size=7, padding=3)
        self.block1 = ConvGLUBlock(base_channels, base_channels * 2, kernel=5, padding=2)
        self.block2 = ConvGLUBlock(base_channels * 2, base_channels * 4, kernel=5, padding=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(base_channels * 4, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        # x: (batch, time, channels)
        x = x.permute(0, 2, 1)            # -> (batch, channels, time)
        x = F.relu(self.stem(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.pool(x).squeeze(-1)     # (batch, channels)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

In [422]:
def compute_accuracy(model, dataloader, device='cpu'):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


In [423]:
train_dataset = NinaProDataset(X_train, y_train)
test_dataset = NinaProDataset(X_test, y_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [424]:
def effective_class_weights(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    weights = (1 - beta) / (1 - beta ** counts)
    weights = weights / weights.sum() * len(classes)
    return torch.tensor(weights, dtype=torch.float32)


In [425]:
# device = 'cuda' if torch.cuda.is_available() else 'cpu'

# num_classes = len(np.unique(y_train))
# model = ResGLUClassifier(in_channels=10, num_classes=num_classes).to(device)
# classes = np.unique(y_train)
# #weights = compute_class_weight("balanced", classes=classes, y=y_train)
# #weights = torch.tensor(weights, dtype=torch.float32).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max', factor=0.5, patience=15)

# num_epochs = 200

# for epoch in range(num_epochs):
#     model.train()
#     for X, y in train_loader:
#         X = X.to(device)
#         y = y.to(device)

#         optimizer.zero_grad()
#         logits = model(X)
#         loss = criterion(logits, y)
#         loss.backward()
#         optimizer.step()

#     # Accuracy assessment
#     train_acc = compute_accuracy(model, train_loader, device)
#     test_acc = compute_accuracy(model, test_loader, device)

#     scheduler.step(test_acc)

#     print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")


In [426]:

from sklearn.metrics import classification_report
def get_all_predictions(model, loader, device):
    y_true = []
    y_pred = []
    model.eval()
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            logits = model(X)
            preds = logits.argmax(dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(y.numpy())
    y_pred = medfilt(np.array(y_pred), kernel_size=5)
    return y_true, y_pred

In [427]:
# y_true, y_pred = get_all_predictions(model, test_loader, device)
# print(classification_report(y_true, y_pred))

In [428]:
class AdaERBuffer:
    def __init__(self, max_size=3000):
        self.max_size = max_size
        self.data = []
        self.targets = []
        self.old_logits = []

    def add(self, X, y, logits):
        for i in range(X.size(0)):
            if len(self.data) >= self.max_size:
                self.data.pop(0)
                self.targets.pop(0)
                self.old_logits.pop(0)

            self.data.append(X[i].cpu())
            self.targets.append(int(y[i].cpu()))
            self.old_logits.append(logits[i].cpu())

    def sample_balanced(self, batch_size):
        classes = list(set(self.targets))
        samples_per_class = max(1, batch_size // len(classes))

        idx = []
        for c in classes:
            c_idx = [i for i, t in enumerate(self.targets) if t == c]
            chosen = np.random.choice(c_idx, samples_per_class, replace=True)
            idx.extend(chosen)

        idx = idx[:batch_size]
        np.random.shuffle(idx)

        x = torch.stack([self.data[i] for i in idx])
        y = torch.tensor([self.targets[i] for i in idx])
        old_logits = torch.stack([self.old_logits[i] for i in idx])
        return x, y, old_logits



In [429]:
# Exercise 1
X1_train, y1_train, X1_test, y1_test, _ = preprocessing('s1/S1_A1_E1.mat')
loader1_train = DataLoader(NinaProDataset(X1_train, y1_train), batch_size=64, shuffle=True)
loader1_test  = DataLoader(NinaProDataset(X1_test, y1_test), batch_size=64, shuffle=False)

# Exercise 2
X2_train, y2_train, X2_test, y2_test, _ = preprocessing('s1/S1_A1_E2.mat')
loader2_train = DataLoader(NinaProDataset(X2_train, y2_train), batch_size=64, shuffle=True)
loader2_test  = DataLoader(NinaProDataset(X2_test, y2_test), batch_size=64, shuffle=False)



subject #[[1]]
exercise #[[1]]
0.0     353
5.0      58
3.0      53
1.0      50
8.0      44
6.0      43
4.0      41
12.0     40
10.0     39
2.0      37
7.0      36
9.0      35
11.0     35
Name: count, dtype: int64
0.0     910
3.0      30
1.0      26
10.0     25
5.0      22
12.0     21
4.0      18
6.0      17
7.0      17
9.0      17
2.0      16
8.0      16
11.0     15
Name: count, dtype: int64
subject #[[1]]
exercise #[[2]]
0.0     490
17.0     57
9.0      50
12.0     50
1.0      49
6.0      47
16.0     47
13.0     46
2.0      45
4.0      45
7.0      43
3.0      42
10.0     41
8.0      39
5.0      36
11.0     36
15.0     34
14.0     30
Name: count, dtype: int64
0.0     1302
6.0       26
17.0      24
3.0       22
12.0      22
16.0      21
10.0      20
2.0       19
7.0       19
8.0       19
9.0       19
11.0      19
15.0      19
1.0       18
13.0      17
5.0       15
4.0       14
14.0      11
Name: count, dtype: int64


In [430]:
e1_unique = np.unique(y1_train)
e1_map = {old: new for new, old in enumerate(e1_unique)}

y1_train = np.array([e1_map[y] for y in y1_train])
y1_test  = np.array([e1_map[y] for y in y1_test])
e2_unique = np.unique(y2_train)
e2_map = {old: (new + len(e1_unique)) for new, old in enumerate(e2_unique)}

y2_train = np.array([e2_map[y] for y in y2_train])
y2_test  = np.array([e2_map[y] for y in y2_test])
num_classes = len(e1_unique) + len(e2_unique)



In [431]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model for both exercises (same architecture)
num_classes = len(np.unique(np.concatenate([y1_train, y2_train])))
model = ResGLUClassifier(in_channels=10, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=15)

buffer = AdaERBuffer(max_size=3000)
lambda_kd = 10
batch_size = 64
num_epochs = 50



In [432]:
for epoch in range(num_epochs):
    model.train()
    for X, y in loader1_train:
        X, y = X.to(device), y.to(device)

        logits = model(X)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Fill buffer with E1 data
        with torch.no_grad():
            buffer.add(X, y, logits)


In [ ]:
for epoch in range(num_epochs):
    model.train()
    for X, y in loader2_train:
        X, y = X.to(device), y.to(device)

        # -------------------------
        # 1. New data loss (E2)
        # -------------------------
        logits = model(X)
        loss_ce = criterion(logits, y)

        # Add new data to buffer
        with torch.no_grad():
            buffer.add(X, y, logits)

        # -------------------------
        # 2. Replay (with ratio)
        # -------------------------
        replay_ratio = 2
        loss_kd = 0.0
        valid_replays = 0

        for _ in range(replay_ratio):
            if len(buffer.data) > batch_size:
                bx, by, bold = buffer.sample_balanced(batch_size)
                bx, by, bold = bx.to(device), by.to(device), bold.to(device)

                blogits = model(bx)

                loss_kd += F.kl_div(
                    F.log_softmax(blogits, dim=1),
                    F.softmax(bold, dim=1),
                    reduction='batchmean'
                )
                valid_replays += 1

        # Average KD over replay batches
        if valid_replays > 0:
            loss_kd /= valid_replays
        else:
            loss_kd = 0.0

        # -------------------------
        # 3. Combine losses
        # -------------------------
        loss = loss_ce + lambda_kd * loss_kd

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



    

In [ ]:

y1_true, y1_pred = get_all_predictions(model, loader1_test, device)
print("=== Exercise 1 Classification Report ===")
print(classification_report(y1_true, y1_pred))

y2_true, y2_pred = get_all_predictions(model, loader2_test, device)
print("=== Exercise 2 Classification Report ===")
print(classification_report(y2_true, y2_pred))


=== Exercise 1 Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       910
           1       0.06      0.12      0.08        26
           2       0.00      0.00      0.00        16
           3       0.00      0.00      0.00        30
           4       0.00      0.00      0.00        18
           5       0.00      0.00      0.00        22
           6       0.42      0.47      0.44        17
           7       0.67      0.47      0.55        17
           8       0.00      0.00      0.00        16
           9       0.00      0.00      0.00        17
          10       0.00      0.00      0.00        25
          11       0.00      0.00      0.00        15
          12       0.23      0.48      0.31        21
          14       0.00      0.00      0.00         0
          16       0.00      0.00      0.00         0
          17       0.00      0.00      0.00         0

    accuracy                           

C:\Users\shubh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\shubh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\shubh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classificati

=== Exercise 2 Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1302
           1       0.90      1.00      0.95        18
           2       1.00      0.95      0.97        19
           3       0.55      0.73      0.63        22
           4       0.60      0.21      0.32        14
           5       0.27      0.20      0.23        15
           6       1.00      0.65      0.79        26
           7       1.00      1.00      1.00        19
           8       1.00      0.95      0.97        19
           9       0.90      1.00      0.95        19
          10       0.66      0.95      0.78        20
          11       1.00      0.79      0.88        19
          12       0.79      0.68      0.73        22
          13       0.62      0.94      0.74        17
          14       0.85      1.00      0.92        11
          15       1.00      0.89      0.94        19
          16       0.71      0.95      0